In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from src import scoring

b = scoring.load()
df = pd.read_csv("../data/processed/policies_clean.csv")
tr_ids = pd.read_csv("../data/processed/train_ids.csv")["IDpol"]
train = df[df["IDpol"].isin(tr_ids)].reset_index(drop=True)

s = scoring.score(b, train.sample(20000, random_state=0)[scoring.INPUTS])
print("mean annual rate:", round(s["annual_rate"].mean(), 4), "| mean relativity:", round(s["relativity"].mean(), 3))
print("tier shares:", s["risk_tier"].value_counts(normalize=True).sort_index().round(3).to_dict())
print("rows with errors:", int((s["errors"] != "").sum()), "| rows with warnings:", int((s["warnings"] != "").sum()))

row = train.iloc[0][scoring.INPUTS].to_dict()
print(scoring.score(b, pd.DataFrame([row])).T)
print(scoring.explain(b, row))

bad = {**row, "VehBrand": "B99", "DrivAge": 150}
print(scoring.score(b, pd.DataFrame([bad]))[["annual_rate", "errors", "warnings"]])




mean annual rate: 0.0794 | mean relativity: 0.979
tier shares: {1.0: 0.101, 2.0: 0.098, 3.0: 0.101, 4.0: 0.103, 5.0: 0.108, 6.0: 0.101, 7.0: 0.097, 8.0: 0.098, 9.0: 0.098, 10.0: 0.094}
rows with errors: 0 | rows with warnings: 1687
                                                             0
annual_rate                                           0.166501
relativity                                            2.053351
risk_tier                                                 10.0
errors                                                        
warnings     Brand-new vehicle: calibration for this group ...
{'base_level': 0.05240304269868601, 'drivers': [('VehAge', np.int64(0), 2.4303786044525024), ('VehPower', np.int64(5), 1.1609939231039512), ('VehGas', 'Regular', 1.132561648306674), ('BonusMalus', np.int64(50), 0.8938738532417538)], 'other_factor': 1.1122937789018599, 'annual_rate': 0.16650143366734482}
   annual_rate                                   errors  \
0          NaN  VehBrand='

In [2]:
from src import gbm
samp = train.sample(20000, random_state=0)
s = scoring.score(b, samp[scoring.INPUTS])
direct = b["model"].predict(gbm.make_X(samp.assign(Exposure=1.0), b["levels"], b["features"]))
assert np.allclose(s["annual_rate"], direct)                 # scoring path == model path
print("sample mean:", direct.mean().round(4))
full = b["model"].predict(gbm.make_X(train.assign(Exposure=1.0), b["levels"], b["features"]))
print("full train mean:", full.mean().round(4), "| reference:", round(b["reference_annual_rate"], 4))

print("std of rates:", full.std().round(4), "| SE of a 20,000-row mean:", (full.std() / np.sqrt(20000)).round(5))
for seed in [1, 2, 3]:
    print(seed, b["model"].predict(gbm.make_X(train.sample(20000, random_state=seed).assign(Exposure=1.0),
                                              b["levels"], b["features"])).mean().round(4))

sample mean: 0.0794
full train mean: 0.0811 | reference: 0.0811
std of rates: 0.0675 | SE of a 20,000-row mean: 0.00048
1 0.0804
2 0.0819
3 0.0807


In [3]:
row = train.iloc[0][scoring.INPUTS].to_dict()

e = scoring.explain(b, row)
print(e)
print("shown x other x base:", round(e["base_level"] * np.prod([f for _, _, f in e["drivers"]]) * e["other_factor"], 4))

{'base_level': 0.05240304269868601, 'drivers': [('VehAge', np.int64(0), 2.4303786044525024), ('VehPower', np.int64(5), 1.1609939231039512), ('VehGas', 'Regular', 1.132561648306674), ('BonusMalus', np.int64(50), 0.8938738532417538)], 'other_factor': 1.1122937789018599, 'annual_rate': 0.16650143366734482}
shown x other x base: 0.1665
